# Performance Estimation Results

Columns: `InstanceType, TP, PP, BatchSize, E2ELatency, Throughput`

In [1]:
import json
import glob
import os
import pandas as pd

EST_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data", "estimated")


def load_estimated(est_dir: str = EST_DIR) -> pd.DataFrame:
    """
    Load all estimation JSONs and flatten batch_sweep into a DataFrame.
    """
    files = sorted(glob.glob(os.path.join(est_dir, "est_*.json")))
    rows = []
    for f in files:
        with open(f) as fp:
            d = json.load(fp)
        if not d.get("feasible", False):
            continue
        instance = d["instance_type"]
        tp = d["tp_size"]
        pp = d["pp_size"]
        for entry in d.get("batch_sweep", []):
            rows.append({
                "InstanceType": instance,
                "TP": tp,
                "PP": pp,
                "BatchSize": entry["batch_size"],
                "E2ELatency": round(entry["batch_latency_ms"], 2),
                "Throughput": round(entry["throughput_rps"], 4),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["InstanceType", "TP", "PP", "BatchSize"]).reset_index(drop=True)
    return df


def filter_df(df, instance=None, tp=None, pp=None):
    """Filter by instance type, TP, and/or PP."""
    if instance:
        df = df[df["InstanceType"] == instance]
    if tp:
        df = df[df["TP"] == tp]
    if pp:
        df = df[df["PP"] == pp]
    return df.reset_index(drop=True)


def max_batch_summary(df):
    """One row per (Instance, TP, PP) — max batch only."""
    return (
        df.loc[df.groupby(["InstanceType", "TP", "PP"])["BatchSize"].idxmax()]
        .sort_values(["InstanceType", "TP", "PP"])
        .reset_index(drop=True)
    )

In [2]:
df = load_estimated()
print(f"{len(df)} rows, {df.groupby(['InstanceType','TP','PP']).ngroups} configs")
df

164 rows, 18 configs


,InstanceType,TP,PP,BatchSize,E2ELatency,Throughput
0,g5.48xlarge,2,4,1,15089.77,0.0663
1,g5.48xlarge,2,4,2,12767.21,0.1567
2,g5.48xlarge,2,4,4,8122.09,0.4925
3,g5.48xlarge,2,4,8,9220.98,0.8676
4,g5.48xlarge,2,4,11,10045.15,1.0951
...,...,...,...,...,...,...
159,p5.48xlarge,8,1,128,4555.61,28.0972
160,p5.48xlarge,8,1,256,7907.90,32.3727
161,p5.48xlarge,8,1,512,15272.30,33.5247
162,p5.48xlarge,8,1,1024,30451.65,33.6271


## Max-Batch Summary

In [3]:
max_batch_summary(df)

,InstanceType,TP,PP,BatchSize,E2ELatency,Throughput
0,g5.48xlarge,2,4,11,10045.15,1.0951
1,g5.48xlarge,4,2,30,17708.61,1.6941
2,g5.48xlarge,8,1,33,24157.25,1.3660
3,g6.48xlarge,2,4,11,16301.89,0.6748
4,g6.48xlarge,4,2,30,22518.35,1.3322
5,g6.48xlarge,8,1,33,28720.95,1.1490
6,g6e.48xlarge,1,8,443,29971.79,14.7806
7,g6e.48xlarge,2,4,481,60249.28,7.9835
8,g6e.48xlarge,4,2,499,105483.70,4.7306
9,g6e.48xlarge,8,1,502,190100.52,2.6407


## Filter Example

In [4]:
# Change parameters as needed
filter_df(df, instance="p5.48xlarge", tp=2, pp=4)

,InstanceType,TP,PP,BatchSize,E2ELatency,Throughput
0,p5.48xlarge,2,4,1,1797.54,0.5563
1,p5.48xlarge,2,4,2,1644.47,1.2162
2,p5.48xlarge,2,4,4,1338.35,2.9887
3,p5.48xlarge,2,4,8,1417.82,5.6425
4,p5.48xlarge,2,4,16,1576.76,10.1474
5,p5.48xlarge,2,4,32,1894.63,16.8899
6,p5.48xlarge,2,4,64,2534.37,25.2528
7,p5.48xlarge,2,4,128,3813.85,33.5619
8,p5.48xlarge,2,4,256,6368.82,40.1958
9,p5.48xlarge,2,4,512,12248.30,41.8017
